# Phase 5 — SVM Experiments
Goal: test whether margin-based models (linear, then RBF) beat the Logistic Regression baseline on expected APS cost.
Baseline to beat (honest, fold-wise): balanced LogReg + signed-log = 51,930.

In [2]:
%load_ext autoreload
%autoreload 2

import sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))

from src.data_loader import load_aps_data
from src.preprocessing import build_linear_preprocessor, make_cv_splitter
from src.evaluation import best_cost_threshold

DATA_DIR = PROJECT_ROOT / "dataset"
X,y = load_aps_data(DATA_DIR/"aps_failure_training_set.csv")
cv = make_cv_splitter()
print(X.shape, dict(y.value_counts()))

(60000, 170) {0: np.int64(59000), 1: np.int64(1000)}


In [10]:
from numpy import size
rng = np.random.default_rng(0)
y_demo = rng.integers(0,2,2000)
s_demo = rng.normal(size=2000) + 3*y_demo-1.5
print(best_cost_threshold(y_demo,s_demo))

{'tn': 653, 'fp': 304, 'fn': 2, 'tp': 1041, 'aps_cost': 4040, 'threshold': -1.0890802948302896, 'cost_per_truck': 2.02}


In [12]:
from numpy import record
from matplotlib import use
def make_linear_svm(C,class_weight):
    return Pipeline([
        ("prep", build_linear_preprocessor(use_log=True, drop_duplicates=True)),
        ("svm", LinearSVC(C=C, class_weight=class_weight, dual=False, max_iter=2000, random_state=42))
    ])

def oof_scores(model, X,y,cv):
    """Out-of-fold decision scores. Each row is scored by a model that never saw it."""
    scores = np.zeros(len(y))
    n_warn, t0=0, time.time()
    for tr, va in cv.split(X,y):
        m = clone(model)
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            m.fit(X.iloc[tr], y.iloc[tr])
        n_warn += any(issubclass(x.category, ConvergenceWarning) for x in w)
        scores[va] = m.decision_function(X.iloc[va])
    return scores, n_warn, time.time() - t0


def honest_cost(y, scores, cv):
    """Per fold: pick the threshold on the OTHER folds' scores, score it on this fold."""
    y_arr = np.asarray(y)
    total, FP, FN = 0, 0, 0
    for tr, va in cv.split(np.zeros(len(y_arr)), y_arr):
        t = best_cost_threshold(y_arr[tr], scores[tr])["threshold"]
        pred = scores[va] >= t
        fp = int((pred & (y_arr[va] == 0)).sum())
        fn = int((~pred & (y_arr[va] == 1)).sum())
        FP, FN, total = FP + fp, FN + fn, total + 10 * fp + 500 * fn
    return total, FP, FN

In [13]:
s, n_warn, secs = oof_scores(make_linear_svm(1.0, "balanced"), X, y, cv)
print("warn folds:", n_warn, "| seconds:", round(secs, 1))
print("ROC-AUC:", round(roc_auc_score(y, s), 4), "| PR-AUC:", round(average_precision_score(y, s), 4))
print(best_cost_threshold(y, s))
print("honest:", honest_cost(y, s, cv))

warn folds: 0 | seconds: 120.5
ROC-AUC: 0.9827 | PR-AUC: 0.7747
{'tn': 56895, 'fp': 2105, 'fn': 62, 'tp': 938, 'aps_cost': 52050, 'threshold': -0.03505789149988825, 'cost_per_truck': 0.8675}
honest: (54640, 2114, 67)


In [16]:
all_scores = {}   # (C, class_weight) -> out-of-fold scores

extra = [(C, None) for C in [3.0, 10.0, 30.0, 100.0]] + \
        [(C, "balanced") for C in [0.001, 0.003]]

for C, cw in extra:
    s, n_warn, secs = oof_scores(make_linear_svm(C, cw), X, y, cv)
    all_scores[(C, cw)] = s
    tuned = best_cost_threshold(y, s)
    h_cost, h_fp, h_fn = honest_cost(y, s, cv)
    rows.append({"model": "LinearSVC", "C": C, "class_weight": cw,
                 "warn_folds": n_warn, "seconds": round(secs, 1),
                 "roc_auc": roc_auc_score(y, s),
                 "pr_auc": average_precision_score(y, s),
                 "tuned_cost": tuned["aps_cost"],
                 "honest_cost": h_cost, "honest_fp": h_fp, "honest_fn": h_fn})

svm_linear_results = pd.DataFrame(rows).sort_values("honest_cost").reset_index(drop=True)
svm_linear_results.round(4)

,model,C,class_weight,warn_folds,seconds,roc_auc,pr_auc,tuned_cost,honest_cost,honest_fp,honest_fn
0,LinearSVC,1.000,None,0,57.2,0.9855,0.8371,44640,45100,2110,48
1,LinearSVC,0.003,balanced,0,148.0,0.9862,0.7772,44980,45450,2245,46
2,LinearSVC,30.000,None,0,212.6,0.9851,0.8345,45660,46110,1811,56
3,LinearSVC,100.000,None,0,160.8,0.9850,0.8342,45660,46150,1815,56
4,LinearSVC,0.100,None,0,51.0,0.9860,0.8367,45710,46180,1868,55
5,LinearSVC,10.000,None,0,117.3,0.9851,0.8350,45760,47180,1968,55
6,LinearSVC,3.000,None,0,55.1,0.9853,0.8361,45930,48570,2257,52
7,LinearSVC,0.001,balanced,0,151.7,0.9860,0.7630,45570,49130,2713,44
8,LinearSVC,0.010,balanced,0,77.1,0.9859,0.7842,47310,50040,2104,58
9,LinearSVC,0.100,balanced,0,80.2,0.9838,0.7807,50270,53830,2033,67


In [15]:
svm_linear_results.to_csv(PROJECT_ROOT / "reports" / "results" / "phase5_linear_svm_results.csv", index=False)

warn folds: 0 | seconds: 135.7
ROC-AUC: 0.9827 | PR-AUC: 0.7747
{'tn': 56895, 'fp': 2105, 'fn': 62, 'tp': 938, 'aps_cost': 52050, 'threshold': -0.03505789149988825, 'cost_per_truck': 0.8675}
honest: (54640, 2114, 67)


In [17]:
from sklearn.linear_model import LogisticRegression

def make_logreg(class_weight):
    return Pipeline([
        ("prep", build_linear_preprocessor(use_log=True, drop_duplicates=True)),
        ("lr", LogisticRegression(max_iter=500, solver="lbfgs", tol=1e-3,
                                  random_state=42, class_weight=class_weight)),
    ])

def fold_costs(y, scores, cv):
    """Honest cost for each fold separately."""
    y_arr, out = np.asarray(y), []
    for tr, va in cv.split(np.zeros(len(y_arr)), y_arr):
        t = best_cost_threshold(y_arr[tr], scores[tr])["threshold"]
        pred = scores[va] >= t
        fp = int((pred & (y_arr[va] == 0)).sum())
        fn = int((~pred & (y_arr[va] == 1)).sum())
        out.append(10 * fp + 500 * fn)
    return np.array(out)

svm_s, _, _ = oof_scores(make_linear_svm(1.0, None), X, y, cv)
lr_u, _, _  = oof_scores(make_logreg(None), X, y, cv)
lr_b, _, _  = oof_scores(make_logreg("balanced"), X, y, cv)

fc = pd.DataFrame({
    "SVM C=1 unweighted": fold_costs(y, svm_s, cv),
    "LogReg unweighted":  fold_costs(y, lr_u, cv),
    "LogReg balanced":    fold_costs(y, lr_b, cv),
}, index=[f"fold {i}" for i in range(1, 6)])
fc.loc["total"] = fc.sum()
fc

,SVM C=1 unweighted,LogReg unweighted,LogReg balanced
fold 1,8920,8050,9820
fold 2,10650,10830,12070
fold 3,8480,9660,9350
fold 4,8420,10030,11100
fold 5,8630,10180,9590
total,45100,48750,51930
